# K-Means Clustering: Complete Advanced Implementation & Analysis

**Objective**: Master K-Means from first principles through advanced applications, performance optimization, and interview preparation.

---

## Part 1: Advanced Theory & Mathematical Foundations

### K-Means Objective Function (Within-Cluster Sum of Squares)

$$J = \sum_{i=1}^{n} \sum_{j=1}^{k} r_{ij} ||x_i - \mu_j||^2$$

Where:
- $x_i$ = data point i (d-dimensional feature vector)
- $\mu_j$ = centroid of cluster j (d-dimensional mean vector)
- $r_{ij}$ = assignment indicator (1 if point i belongs to cluster j, 0 otherwise)
- $n$ = total number of points
- $k$ = number of clusters
- Goal: Minimize within-cluster variance (inertia)

This objective is also called the **inertia** or **distortion**. Lower J means tighter, more cohesive clusters.

### Update Rules (Derivation from Lagrangian)

To minimize J, we alternate between two optimization steps:

**Centroid Update (M-Step - Maximization)**:
$$\mu_j = \frac{\sum_{i=1}^{n} r_{ij} x_i}{\sum_{i=1}^{n} r_{ij}}$$

Each centroid moves to the mean position of all points assigned to it. This minimizes the sum of squared distances for fixed assignments.

**Assignment Rule (E-Step - Expectation)**:
$$r_{ij} = \begin{cases} 1 & \text{if } j = \arg\min_k ||x_i - \mu_k||^2 \\ 0 & \text{otherwise} \end{cases}$$

Assign each point to the cluster with the nearest centroid. This minimizes the objective for fixed centroids.

### Convergence Analysis

**Theorem**: K-Means converges to a local minimum of the objective function.

**Proof Sketch**:
1. Assignment step decreases (or keeps same) objective: $J^{(t+1)} \leq J^{(t)}$
2. Centroid update step decreases (or keeps same) objective: $J^{(t+2)} \leq J^{(t+1)}$
3. Since J is bounded below (≥ 0) and non-increasing, it must converge
4. At convergence: no point changes cluster (fixed point)

**Important**: Convergence to local optimum is guaranteed, but global optimum not guaranteed. Multiple initializations help find better local optima.

### K-Means++ Initialization

Standard random initialization can lead to poor local optima. K-Means++ (Arthur & Vassilvitskii, 2007) is better:

1. **First centroid**: Choose randomly from data points
2. **Subsequent centroids**: Choose with probability proportional to squared distance from nearest existing centroid
   $$P(\text{choose point } x_i) \propto \min_j ||x_i - \mu_j||^2$$

**Key Insight**: This spreads initial centroids far apart, reducing chance of poor local optima. Empirically reduces iterations and improves cluster quality.

### Complexity Analysis

**Time Complexity: O(n·k·d·i)**
- n = number of points
- k = number of clusters
- d = number of dimensions
- i = number of iterations (typically 10-100, much smaller than n)

**Space Complexity: O(n·d + k·d)**
- O(n·d) for storing data
- O(k·d) for storing centroids

Compared to GMM: O(n·k·d²·i) - quadratic in dimensions
Compared to DBSCAN: O(n²) - quadratic in points

### Why K-Means Assumes Spherical Clusters

The Euclidean distance metric and centroid-based objective naturally prefer:
- **Spherical shapes**: Distance to center is uniform in all directions
- **Similar sizes**: Large clusters pull centroid away from boundary
- **Continuous features**: Breaks with categorical or ordinal data

**Mathematical insight**: For non-spherical clusters (e.g., elliptical), the centroid-based assignment is suboptimal. Points on the elongated axis far from centroid still assigned to cluster, increasing inertia.

### Local vs Global Optimality

K-Means solves a non-convex optimization problem. The objective function J(r, μ) is:
- **Convex** in μ (for fixed assignments) → global optimum for centroids
- **Convex** in r (for fixed centroids) → global optimum for assignments
- **Non-convex** jointly → local optima possible

**Example of poor local optimum**:
- 100 points: 99 tight in one region, 1 outlier far away
- True clustering: 99 + 1
- Poor local optimum: Split 99 into two clusters of 50 each to minimize distance from outlier centroid

This is why we use K-Means++ and multiple initializations.

In [ ]:
# Part 2: From-Scratch Implementation with Advanced Features
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import time
from scipy.spatial.distance import cdist

# Set random seed for reproducibility
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

class KMeansFromScratch:
    """Advanced K-Means clustering algorithm implemented from scratch"""
    
    def __init__(self, k=3, max_iterations=100, tol=1e-4, init_method='kmeans++', random_state=42):
        """
        Initialize K-Means
        
        Parameters:
        -----------
        k : int
            Number of clusters
        max_iterations : int
            Maximum number of iterations
        tol : float
            Tolerance for convergence (centroid movement)
        init_method : str
            'random' or 'kmeans++' for initialization
        random_state : int
            Random seed
        """
        self.k = k
        self.max_iterations = max_iterations
        self.tol = tol
        self.init_method = init_method
        self.random_state = random_state
        self.centroids = None
        self.labels = None
        self.inertia_history = []
        self.centroid_movement_history = []
        self.n_iterations = 0
        self.runtime = 0
        
    def initialize_centroids_random(self, X):
        """Randomly select K points as initial centroids"""
        np.random.seed(self.random_state)
        random_indices = np.random.choice(X.shape[0], self.k, replace=False)
        return X[random_indices]
    
    def initialize_centroids_kmeans_plus_plus(self, X):
        """K-Means++ initialization: spread centroids far apart"""
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        
        # First centroid: random point
        first_idx = np.random.randint(n_samples)
        centroids = [X[first_idx]]
        
        # Subsequent centroids: choose with probability proportional to squared distance
        for _ in range(1, self.k):
            # Compute distances to nearest centroid
            distances = np.array([min([np.linalg.norm(x - c)**2 for c in centroids]) for x in X])
            
            # Choose next centroid with probability proportional to distance squared
            probabilities = distances / distances.sum()
            cumulative_probs = np.cumsum(probabilities)
            r = np.random.random()
            next_idx = np.searchsorted(cumulative_probs, r)
            centroids.append(X[next_idx])
        
        return np.array(centroids)
    
    def assign_clusters(self, X):
        """Assign each point to nearest centroid"""
        # Compute Euclidean distances from each point to all centroids
        distances = np.sqrt(((X - self.centroids[:, np.newaxis])**2).sum(axis=2))
        # Assign to nearest centroid
        return np.argmin(distances, axis=0)
    
    def update_centroids(self, X, labels):
        """Update centroids as mean of assigned points"""
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(self.k)])
        return new_centroids
    
    def compute_inertia(self, X, labels):
        """Compute within-cluster sum of squares (inertia)"""
        inertia = 0
        for i in range(self.k):
            cluster_points = X[labels == i]
            if len(cluster_points) > 0:
                inertia += np.sum((cluster_points - self.centroids[i])**2)
        return inertia
    
    def compute_silhouette_coefficient(self, X, labels):
        """Compute silhouette coefficient for cluster quality (-1 to 1)"""
        if len(np.unique(labels)) == 1:
            return -1  # Undefined for single cluster
        
        silhouette_scores = []
        for i in range(X.shape[0]):
            # a(i): mean distance to points in same cluster
            same_cluster_points = X[labels == labels[i]]
            a_i = np.mean([np.linalg.norm(X[i] - p) for p in same_cluster_points if not np.array_equal(X[i], p) or len(same_cluster_points) > 1])
            
            # b(i): mean distance to points in nearest other cluster
            b_i = np.inf
            for cluster_id in np.unique(labels):
                if cluster_id != labels[i]:
                    other_cluster_points = X[labels == cluster_id]
                    mean_dist = np.mean([np.linalg.norm(X[i] - p) for p in other_cluster_points])
                    b_i = min(b_i, mean_dist)
            
            # s(i) = (b(i) - a(i)) / max(a(i), b(i))
            if max(a_i, b_i) > 0:
                s_i = (b_i - a_i) / max(a_i, b_i)
            else:
                s_i = 0
            silhouette_scores.append(s_i)
        
        return np.mean(silhouette_scores) if silhouette_scores else 0
    
    def fit(self, X):
        """Fit K-Means on data"""
        start_time = time.time()
        
        # Initialize centroids
        if self.init_method == 'kmeans++':
            self.centroids = self.initialize_centroids_kmeans_plus_plus(X)
        else:
            self.centroids = self.initialize_centroids_random(X)
        
        # Iterate until convergence
        for iteration in range(self.max_iterations):
            # Store old centroids for convergence check
            old_centroids = self.centroids.copy()
            
            # Assign step
            self.labels = self.assign_clusters(X)
            
            # Update step
            self.centroids = self.update_centroids(X, self.labels)
            
            # Compute inertia
            inertia = self.compute_inertia(X, self.labels)
            self.inertia_history.append(inertia)
            
            # Compute centroid movement
            centroid_movement = np.sum([np.linalg.norm(self.centroids[i] - old_centroids[i]) for i in range(self.k)])
            self.centroid_movement_history.append(centroid_movement)
            
            # Check convergence
            if centroid_movement < self.tol:
                self.n_iterations = iteration + 1
                break
        
        self.n_iterations = iteration + 1
        self.runtime = time.time() - start_time
        self.inertia = inertia
        
        return self
    
    def predict(self, X):
        """Predict cluster labels for new points"""
        distances = np.sqrt(((X - self.centroids[:, np.newaxis])**2).sum(axis=2))
        return np.argmin(distances, axis=0)
    
    def get_stats(self):
        """Return summary statistics"""
        return {
            'inertia': self.inertia,
            'n_iterations': self.n_iterations,
            'runtime_ms': self.runtime * 1000,
            'silhouette': self.compute_silhouette_coefficient if hasattr(self, 'compute_silhouette_coefficient') else None
        }

# Generate multiple datasets to test
print(f"\n{'='*70}")
print("GENERATING TEST DATASETS")
print(f"{'='*70}")

# Dataset 1: Spherical clusters (K-Means ideal)
X_spherical, y_spherical = make_blobs(n_samples=300, centers=3, n_features=2, random_state=42, cluster_std=0.8)
X_spherical_scaled = StandardScaler().fit_transform(X_spherical)
print("Dataset 1: Spherical clusters (300 points, 3 centers)")

# Dataset 2: Non-convex (crescent) - K-Means will struggle
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
X_moons_scaled = StandardScaler().fit_transform(X_moons)
print("Dataset 2: Non-convex moons (300 points, 2 crescents)")

# Dataset 3: Circles (nested) - K-Means will struggle
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
X_circles_scaled = StandardScaler().fit_transform(X_circles)
print("Dataset 3: Nested circles (300 points, 2 rings)")

print("\n" + "="*70)
print("FITTING FROM-SCRATCH K-MEANS ON SPHERICAL DATA")
print("="*70)

# Compare random vs K-Means++ initialization
for init_method in ['random', 'kmeans++']:
    print(f"\nInitialization: {init_method.upper()}")
    results_init = []
    for run in range(3):
        kmeans_scratch = KMeansFromScratch(k=3, max_iterations=100, init_method=init_method, random_state=42+run)
        kmeans_scratch.fit(X_spherical_scaled)
        results_init.append({
            'inertia': kmeans_scratch.inertia,
            'iterations': kmeans_scratch.n_iterations,
            'runtime_ms': kmeans_scratch.runtime * 1000
        })
        print(f"  Run {run+1}: Inertia={kmeans_scratch.inertia:.4f}, Iterations={kmeans_scratch.n_iterations}, Time={kmeans_scratch.runtime*1000:.2f}ms")

# Store best K-Means++ result for later visualization
kmeans_scratch_best = KMeansFromScratch(k=3, max_iterations=100, init_method='kmeans++', random_state=42)
kmeans_scratch_best.fit(X_spherical_scaled)

## Part 3: Advanced Scikit-Learn Implementation with Comparisons

### Why Scikit-Learn K-Means is Better

Scikit-Learn's implementation includes critical optimizations:

1. **K-Means++ Initialization**: Smarter seed selection reducing poor local optima
2. **Multiple Initializations**: Runs algorithm n_init times with different seeds, returns best
3. **Efficient Distance Computation**: Vectorized operations using NumPy/SciPy
4. **Sparse Matrix Support**: Handles sparse data efficiently
5. **Mini-Batch K-Means**: For very large datasets (streaming variant)
6. **Additional Metrics**: Silhouette score, Davies-Bouldin index, etc.

### Performance Characteristics

- **Standard K-Means**: O(n·k·d·i) with vectorization
- **Mini-Batch K-Means**: O(b·k·d·i) where b << n (batch size)
- Memory efficient for large-scale clustering

### Choosing Between Implementations

| Scenario | Recommendation |
|----------|----------------|
| Small dataset (< 10K), learning | Use from-scratch for understanding |
| Medium dataset (10K-1M), production | Use sklearn.cluster.KMeans |
| Very large dataset (> 1M) | Use MiniBatchKMeans |
| Need GPU acceleration | Use cuML (RAPIDS) or TensorFlow/PyTorch |
| Interpretability critical | K-Means with visualizations |
| Speed critical, scalability needed | Mini-batch variants |

In [ ]:
# Part 3: Scikit-Learn Comparison
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print(f"\n{'='*70}")
print("SCIKIT-LEARN K-MEANS ANALYSIS")
print(f"{'='*70}")

# Standard K-Means with different initializations
kmeans_sklearn = KMeans(
    n_clusters=3,
    init='k-means++',
    max_iter=300,
    n_init=10,  # Multiple initializations
    random_state=42
)

start_time = time.time()
labels_sklearn = kmeans_sklearn.fit_predict(X_spherical_scaled)
sklearn_runtime = time.time() - start_time

# Compute multiple quality metrics
sil_score = silhouette_score(X_spherical_scaled, labels_sklearn)
db_score = davies_bouldin_score(X_spherical_scaled, labels_sklearn)
ch_score = calinski_harabasz_score(X_spherical_scaled, labels_sklearn)

print(f"\nStandard K-Means Results:")
print(f"  Inertia: {kmeans_sklearn.inertia_:.4f}")
print(f"  Silhouette Score: {sil_score:.4f} (range: -1 to 1, higher better)")
print(f"  Davies-Bouldin Index: {db_score:.4f} (lower better)")
print(f"  Calinski-Harabasz Score: {ch_score:.4f} (higher better)")
print(f"  N Iterations: {kmeans_sklearn.n_iter_}")
print(f"  Runtime: {sklearn_runtime*1000:.2f}ms")
print(f"  Cluster Sizes: {np.bincount(labels_sklearn)}")

# Mini-Batch K-Means for comparison
print(f"\n{'='*70}")
print("MINI-BATCH K-MEANS (FOR LARGE-SCALE DATA)")
print(f"{'='*70}")

mbkmeans = MiniBatchKMeans(
    n_clusters=3,
    batch_size=30,
    n_init=10,
    random_state=42
)

start_time = time.time()
labels_mb = mbkmeans.fit_predict(X_spherical_scaled)
mb_runtime = time.time() - start_time

sil_mb = silhouette_score(X_spherical_scaled, labels_mb)
db_mb = davies_bouldin_score(X_spherical_scaled, labels_mb)

print(f"\nMini-Batch K-Means Results:")
print(f"  Inertia: {mbkmeans.inertia_:.4f}")
print(f"  Silhouette Score: {sil_mb:.4f}")
print(f"  Davies-Bouldin Index: {db_mb:.4f}")
print(f"  N Iterations: {mbkmeans.n_iter_}")
print(f"  Runtime: {mb_runtime*1000:.2f}ms")
print(f"  Speedup: {sklearn_runtime/mb_runtime:.2f}x faster")
print(f"  Cluster Sizes: {np.bincount(labels_mb)}")

# Compare from-scratch vs sklearn
print(f"\n{'='*70}")
print("COMPARISON: FROM-SCRATCH vs SKLEARN")
print(f"{'='*70}")
print(f"\n{'Method':<20} {'Inertia':<12} {'Runtime(ms)':<12} {'Iterations':<10}")
print(f"{'-'*70}")
print(f"{'From-Scratch':<20} {kmeans_scratch_best.inertia:<12.4f} {kmeans_scratch_best.runtime*1000:<12.2f} {kmeans_scratch_best.n_iterations:<10}")
print(f"{'Sklearn KMeans':<20} {kmeans_sklearn.inertia_:<12.4f} {sklearn_runtime*1000:<12.2f} {kmeans_sklearn.n_iter_:<10}")
print(f"{'Sklearn MiniBatch':<20} {mbkmeans.inertia_:<12.4f} {mb_runtime*1000:<12.2f} {mbkmeans.n_iter_:<10}")
print(f"\nNote: Sklearn's K-Means++ is smarter and typically finds better solutions faster")

## Part 4: Comprehensive Visualizations with Multiple Datasets

### Plot Strategy

We'll visualize K-Means on three different data types:
1. **Spherical Data**: K-Means ideal case (will succeed)
2. **Non-Convex (Moons)**: K-Means difficult case (will fail)
3. **Nested (Circles)**: K-Means very difficult (will fail)

This demonstrates the algorithm's assumptions and limitations clearly.

In [ ]:
# Part 4: Comprehensive Visualizations
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# Row 1: Spherical Data
ax1 = fig.add_subplot(gs[0, 0])
scatter1 = ax1.scatter(X_spherical_scaled[:, 0], X_spherical_scaled[:, 1], c=labels_sklearn, cmap='viridis', alpha=0.6, s=50)
ax1.scatter(kmeans_sklearn.cluster_centers_[:, 0], kmeans_sklearn.cluster_centers_[:, 1], 
           c='red', marker='X', s=300, edgecolors='black', linewidths=2, label='Centroids')
ax1.set_xlabel('Feature 1', fontsize=10)
ax1.set_ylabel('Feature 2', fontsize=10)
ax1.set_title('K-Means on Spherical Data (SUCCEEDS)', fontsize=11, fontweight='bold', color='green')
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Row 1: Moons Data
ax2 = fig.add_subplot(gs[0, 1])
kmeans_moons = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
labels_moons = kmeans_moons.fit_predict(X_moons_scaled)
scatter2 = ax2.scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=labels_moons, cmap='viridis', alpha=0.6, s=50)
ax2.scatter(kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1], 
           c='red', marker='X', s=300, edgecolors='black', linewidths=2, label='Centroids')
ax2.set_xlabel('Feature 1', fontsize=10)
ax2.set_ylabel('Feature 2', fontsize=10)
ax2.set_title('K-Means on Moons Data (FAILS)', fontsize=11, fontweight='bold', color='red')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=ax2, label='Cluster')

# Row 1: Circles Data
ax3 = fig.add_subplot(gs[0, 2])
kmeans_circles = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
labels_circles = kmeans_circles.fit_predict(X_circles_scaled)
scatter3 = ax3.scatter(X_circles_scaled[:, 0], X_circles_scaled[:, 1], c=labels_circles, cmap='viridis', alpha=0.6, s=50)
ax3.scatter(kmeans_circles.cluster_centers_[:, 0], kmeans_circles.cluster_centers_[:, 1], 
           c='red', marker='X', s=300, edgecolors='black', linewidths=2, label='Centroids')
ax3.set_xlabel('Feature 1', fontsize=10)
ax3.set_ylabel('Feature 2', fontsize=10)
ax3.set_title('K-Means on Circles Data (FAILS)', fontsize=11, fontweight='bold', color='red')
ax3.legend()
ax3.grid(True, alpha=0.3)
plt.colorbar(scatter3, ax=ax3, label='Cluster')

# Row 2: Inertia Convergence
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(kmeans_scratch_best.inertia_history, 'bo-', linewidth=2, markersize=6, label='From-Scratch')
ax4.set_xlabel('Iteration', fontsize=10)
ax4.set_ylabel('Inertia (WCSS)', fontsize=10)
ax4.set_title('K-Means Convergence: Inertia', fontsize=11, fontweight='bold')
ax4.grid(True, alpha=0.3)
ax4.legend()

# Row 2: Centroid Movement
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(kmeans_scratch_best.centroid_movement_history, 'gs-', linewidth=2, markersize=6)
ax5.set_xlabel('Iteration', fontsize=10)
ax5.set_ylabel('Centroid Movement', fontsize=10)
ax5.set_title('K-Means Convergence: Centroid Movement', fontsize=11, fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.axhline(y=1e-4, color='r', linestyle='--', linewidth=1, label='Tolerance')
ax5.legend()

# Row 2: Initialization Impact
ax6 = fig.add_subplot(gs[1, 2])
init_results = []
for init_method in ['random', 'kmeans++']:
    inertias = []
    for run in range(10):
        kmeans_temp = KMeans(n_clusters=3, init=init_method, n_init=1, random_state=42+run, max_iter=10)
        kmeans_temp.fit(X_spherical_scaled)
        inertias.append(kmeans_temp.inertia_)
    init_results.append(inertias)

box_data = [init_results[0], init_results[1]]
ax6.boxplot(box_data, labels=['Random', 'K-Means++'])
ax6.set_ylabel('Final Inertia', fontsize=10)
ax6.set_title('Initialization Method Impact (10 runs)', fontsize=11, fontweight='bold')
ax6.grid(True, alpha=0.3, axis='y')

# Row 3: Elbow Method (Finding Optimal K)
ax7 = fig.add_subplot(gs[2, 0])
K_range = range(1, 11)
inertias = []
sil_scores = []

for k in K_range:
    kmeans_k = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans_k.fit(X_spherical_scaled)
    inertias.append(kmeans_k.inertia_)
    if k > 1:
        sil_scores.append(silhouette_score(X_spherical_scaled, kmeans_k.labels_))
    else:
        sil_scores.append(0)

ax7.plot(K_range, inertias, 'mo-', linewidth=2, markersize=8, label='Inertia')
ax7.axvline(x=3, color='r', linestyle='--', linewidth=2, alpha=0.7, label='True K=3')
ax7.set_xlabel('Number of Clusters (K)', fontsize=10)
ax7.set_ylabel('Inertia', fontsize=10)
ax7.set_title('Elbow Method for K Selection', fontsize=11, fontweight='bold')
ax7.grid(True, alpha=0.3)
ax7.legend()

# Row 3: Silhouette Analysis
ax8 = fig.add_subplot(gs[2, 1])
ax8.plot(K_range, sil_scores, 'co-', linewidth=2, markersize=8)
ax8.axvline(x=3, color='r', linestyle='--', linewidth=2, alpha=0.7, label='True K=3')
ax8.set_xlabel('Number of Clusters (K)', fontsize=10)
ax8.set_ylabel('Silhouette Score', fontsize=10)
ax8.set_title('Silhouette Score Analysis', fontsize=11, fontweight='bold')
ax8.grid(True, alpha=0.3)
ax8.legend()
ax8.set_ylim([-0.2, 1.0])

# Row 3: Runtime Comparison
ax9 = fig.add_subplot(gs[2, 2])
methods = ['From-Scratch\n(Sklearn K++)', 'Sklearn\nKMeans', 'Sklearn\nMiniBatch']
runtimes = [kmeans_scratch_best.runtime*1000, sklearn_runtime*1000, mb_runtime*1000]
colors_bar = ['skyblue', 'lightcoral', 'lightgreen']
bars = ax9.bar(methods, runtimes, color=colors_bar, edgecolor='black', linewidth=1.5)
ax9.set_ylabel('Runtime (milliseconds)', fontsize=10)
ax9.set_title('Algorithm Runtime Comparison', fontsize=11, fontweight='bold')
ax9.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, runtime in zip(bars, runtimes):
    height = bar.get_height()
    ax9.text(bar.get_x() + bar.get_width()/2., height,
            f'{runtime:.1f}ms',
            ha='center', va='bottom', fontsize=9)

plt.suptitle('K-Means Clustering: Comprehensive Analysis', fontsize=14, fontweight='bold', y=0.995)
plt.savefig('kmeans_comprehensive_visualizations.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Comprehensive visualizations created successfully!")

## Part 5: Advanced Hyperparameter Experiments with Cross-Validation

### Experimental Design

We systematically vary:
1. **Number of clusters (K)**: 1-10
2. **Initialization method**: 'random', 'k-means++'
3. **Number of initializations (n_init)**: 1, 5, 10, 20
4. **Data scaling**: Scaled vs unscaled features
5. **Distance metric impact**: How feature scaling affects clustering

### Performance Metrics

1. **Inertia (WCSS)**: Sum of squared distances; lower is better
2. **Silhouette Score**: Measures cluster cohesion; range [-1, 1], higher better
3. **Davies-Bouldin Index**: Ratio of within to between cluster distances; lower better
4. **Calinski-Harabasz Score**: Ratio of between to within cluster distances; higher better
5. **Computation Time**: Runtime in milliseconds

In [ ]:
# Part 5: Advanced Hyperparameter Experiments
print(f"\n{'='*80}")
print("HYPERPARAMETER EXPERIMENT 1: K Values and Quality Metrics")
print(f"{'='*80}")

K_range = range(1, 11)
results_k = []

for k in K_range:
    kmeans_k = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels_k = kmeans_k.fit_predict(X_spherical_scaled)
    
    sil = silhouette_score(X_spherical_scaled, labels_k) if k > 1 else -1
    db = davies_bouldin_score(X_spherical_scaled, labels_k) if k > 1 else np.inf
    ch = calinski_harabasz_score(X_spherical_scaled, labels_k) if k > 1 else 0
    
    results_k.append({
        'K': k,
        'Inertia': kmeans_k.inertia_,
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch
    })
    
    print(f"K={k:2d} | Inertia={kmeans_k.inertia_:8.2f} | Silhouette={sil:6.3f} | DB={db:6.3f} | CH={ch:7.1f}")

print(f"\n{'='*80}")
print("HYPERPARAMETER EXPERIMENT 2: Initialization Method (10 runs each)")
print(f"{'='*80}")

init_methods = ['random', 'k-means++']
for method in init_methods:
    print(f"\nInitialization: {method.upper()}")
    inertias_runs = []
    for run in range(10):
        kmeans_init = KMeans(n_clusters=3, init=method, n_init=1, random_state=42+run, max_iter=100)
        kmeans_init.fit(X_spherical_scaled)
        inertias_runs.append(kmeans_init.inertia_)
    
    print(f"  Mean Inertia: {np.mean(inertias_runs):.4f}")
    print(f"  Std Dev: {np.std(inertias_runs):.4f}")
    print(f"  Min: {np.min(inertias_runs):.4f}")
    print(f"  Max: {np.max(inertias_runs):.4f}")
    print(f"  Range: {np.max(inertias_runs) - np.min(inertias_runs):.4f}")

print(f"\n{'='*80}")
print("HYPERPARAMETER EXPERIMENT 3: Number of Initializations (n_init)")
print(f"{'='*80}")

n_init_values = [1, 5, 10, 20, 50]
for n_init_val in n_init_values:
    inertias_repeated = []
    times_repeated = []
    
    for rep in range(5):
        start = time.time()
        kmeans_ni = KMeans(n_clusters=3, init='k-means++', n_init=n_init_val, random_state=42+rep)
        kmeans_ni.fit(X_spherical_scaled)
        inertias_repeated.append(kmeans_ni.inertia_)
        times_repeated.append(time.time() - start)
    
    print(f"n_init={n_init_val:2d} | Mean Inertia={np.mean(inertias_repeated):.4f} (±{np.std(inertias_repeated):.4f}) | Avg Time={np.mean(times_repeated)*1000:.2f}ms")

print(f"\n{'='*80}")
print("HYPERPARAMETER EXPERIMENT 4: Feature Scaling Impact")
print(f"{'='*80}")

print("\nScaled Features:")
kmeans_scaled = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_scaled = kmeans_scaled.fit_predict(X_spherical_scaled)
sil_scaled = silhouette_score(X_spherical_scaled, labels_scaled)
print(f"  Inertia: {kmeans_scaled.inertia_:.4f}")
print(f"  Silhouette: {sil_scaled:.4f}")

print("\nUnscaled Features:")
kmeans_unscaled = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_unscaled = kmeans_unscaled.fit_predict(X_spherical)  # Use original, unscaled
sil_unscaled = silhouette_score(X_spherical, labels_unscaled)
print(f"  Inertia: {kmeans_unscaled.inertia_:.4f}")
print(f"  Silhouette: {sil_unscaled:.4f}")
print(f"\n  Note: Unscaled data with different feature ranges leads to bias!")

In [ ]:
# Part 5: Hyperparameter Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Plot 1: K vs Multiple Metrics
ax1 = axes[0, 0]
k_vals = [r['K'] for r in results_k]
inertias = [r['Inertia'] for r in results_k]
ax1.plot(k_vals, inertias, 'b-o', linewidth=2, markersize=8)
ax1.axvline(x=3, color='r', linestyle='--', linewidth=2, alpha=0.7)
ax1.set_xlabel('Number of Clusters (K)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Inertia', fontsize=11)
ax1.set_title('K vs Inertia (Elbow Method)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot 2: K vs Silhouette Score
ax2 = axes[0, 1]
sil_vals = [r['Silhouette'] for r in results_k]
ax2.plot(k_vals, sil_vals, 'g-s', linewidth=2, markersize=8)
ax2.axvline(x=3, color='r', linestyle='--', linewidth=2, alpha=0.7)
ax2.set_xlabel('Number of Clusters (K)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.set_title('K vs Silhouette Score', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([-0.2, 1.0])

# Plot 3: K vs Davies-Bouldin Index
ax3 = axes[0, 2]
db_vals = [r['Davies-Bouldin'] for r in results_k]
ax3.plot(k_vals, db_vals, 'c-^', linewidth=2, markersize=8)
ax3.axvline(x=3, color='r', linestyle='--', linewidth=2, alpha=0.7)
ax3.set_xlabel('Number of Clusters (K)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Davies-Bouldin Index', fontsize=11)
ax3.set_title('K vs Davies-Bouldin (Lower Better)', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Plot 4: Initialization Method Comparison
ax4 = axes[1, 0]
init_data_random = []
init_data_kmpp = []

for run in range(10):
    kmeans_r = KMeans(n_clusters=3, init='random', n_init=1, random_state=42+run)
    kmeans_r.fit(X_spherical_scaled)
    init_data_random.append(kmeans_r.inertia_)
    
    kmeans_pp = KMeans(n_clusters=3, init='k-means++', n_init=1, random_state=42+run)
    kmeans_pp.fit(X_spherical_scaled)
    init_data_kmpp.append(kmeans_pp.inertia_)

box_data = [init_data_random, init_data_kmpp]
ax4.boxplot(box_data, labels=['Random Init', 'K-Means++ Init'])
ax4.set_ylabel('Final Inertia', fontsize=11)
ax4.set_title('Initialization Method Impact', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Plot 5: n_init Impact on Stability
ax5 = axes[1, 1]
n_init_vals = [1, 5, 10, 20]
n_init_results = []

for n_init_val in n_init_vals:
    inertias_ni = []
    for rep in range(10):
        kmeans_ni = KMeans(n_clusters=3, init='k-means++', n_init=n_init_val, random_state=42+rep)
        kmeans_ni.fit(X_spherical_scaled)
        inertias_ni.append(kmeans_ni.inertia_)
    n_init_results.append(inertias_ni)

ax5.boxplot(n_init_results, labels=[f'{v}' for v in n_init_vals])
ax5.set_xlabel('Number of Initializations (n_init)', fontsize=11, fontweight='bold')
ax5.set_ylabel('Final Inertia', fontsize=11)
ax5.set_title('n_init Impact on Stability', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='y')

# Plot 6: Feature Scaling Impact
ax6 = axes[1, 2]
methods_scale = ['Scaled\nFeatures', 'Unscaled\nFeatures']
inertias_scale = [kmeans_scaled.inertia_, kmeans_unscaled.inertia_]
sils_scale = [sil_scaled, sil_unscaled]

width = 0.35
x_pos = np.arange(len(methods_scale))

ax6_twin = ax6.twinx()
ax6.bar(x_pos - width/2, inertias_scale, width, label='Inertia', color='steelblue', alpha=0.8)
ax6_twin.bar(x_pos + width/2, sils_scale, width, label='Silhouette', color='coral', alpha=0.8)

ax6.set_xlabel('Feature Scaling', fontsize=11, fontweight='bold')
ax6.set_ylabel('Inertia', fontsize=11, color='steelblue')
ax6_twin.set_ylabel('Silhouette Score', fontsize=11, color='coral')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(methods_scale)
ax6.set_title('Feature Scaling Impact', fontsize=12, fontweight='bold')
ax6.tick_params(axis='y', labelcolor='steelblue')
ax6_twin.tick_params(axis='y', labelcolor='coral')
ax6.grid(True, alpha=0.3, axis='y')
ax6.legend(loc='upper left')
ax6_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig('kmeans_hyperparameter_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Hyperparameter analysis visualizations created!")

## Part 6: Interview Preparation - Advanced Questions & Answers

### Question 1: Why K-Means Fails on Non-Convex Data (Deep Dive)

**Technical Answer**:

K-Means minimizes within-cluster sum of squares (WCSS):
$$J = \sum_{i=1}^{n} ||x_i - \mu_{c(i)}||^2$$

This objective is **Euclidean-distance-centric** and **centroid-based**. For a crescent-shaped cluster:

1. **Geometry Problem**: A single point (centroid) cannot represent an elongated curve
2. **Variance Problem**: Points on opposite ends of crescent have high squared distance to centroid
3. **Optimization Solution**: To minimize variance, K-Means splits crescent into multiple spheres
4. **Root Cause**: The algorithm treats variance as the ONLY clustering criterion; ignores connectivity

**Solution**: Use algorithms that consider density (DBSCAN) or spectral properties (Spectral Clustering).

### Question 2: K-Means++ vs Random Initialization (Mathematical Explanation)

**Problem with Random Init**:
- Can place centroids near each other, far from true cluster centers
- Requires many iterations to converge (if at all to good local optimum)
- Different random seeds lead to vastly different results

**K-Means++ Solution**:
$$P(\text{choose } x_i) \propto D(x_i)^2 = [\min_j ||x_i - \mu_j||^2]$$

- First centroid: Uniform random (any point equally likely)
- Subsequent: Probability proportional to squared distance from nearest existing centroid
- Effect: Spreads centroids apart, reducing poor initializations by 8x-10x
- Proof: K-Means++ gives O(log K) approximation to optimal; random gives O(K) worst-case

### Question 3: How to Choose K Without Prior Knowledge

**Method 1: Elbow Method**
- Plot inertia vs K
- Look for "elbow" point where improvement plateaus
- Interpretation: Point of diminishing returns
- Limitation: Elbow can be ambiguous

**Method 2: Silhouette Score**
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$
- Range: [-1, 1]; higher is better
- a(i) = mean distance to points in same cluster (cohesion)
- b(i) = mean distance to nearest other cluster (separation)
- Choose K maximizing average silhouette

**Method 3: Davies-Bouldin Index**
$$DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} \frac{\sigma_i + \sigma_j}{d(\mu_i, \mu_j)}$$
- Lower is better (lower ratio of within to between cluster distance)
- More stable than elbow method

**Method 4: Gap Statistic**
- Compare clustering quality to random data
- Choose K where gap largest
- Most statistically principled

### Question 4: When K-Means is Optimal

K-Means is provably optimal (global minimum) when:
1. Clusters are **perfectly separated** spheres of same radius
2. **No overlapping** between clusters
3. **Equal cluster sizes** (balanced)
4. **Initialization** at true centroids

In practice, K-Means is heuristic (local optimum) because:
- Real data rarely satisfies these conditions
- Problem is NP-hard in general
- Multiple initializations improve but don't guarantee global optimum

### Question 5: Computational Complexity Trade-offs

**K-Means: O(n·k·d·i)**
- Scales linearly with all dimensions except iterations
- Iterations typically 10-100 << n (number of points)
- Practical: 1M points with K=100, d=50, i=20 = 100B operations ≈ 1 second

**GMM: O(n·k·d²·i)**
- Quadratic in dimensions due to covariance computation
- Same 1M points example = 5T operations ≈ 50 seconds (50x slower)
- Better for small d, worse for high-dimensional data

**DBSCAN: O(n²) naive, O(n log n) with spatial indexing**
- Same 1M points: 10^12 comparisons (naive) ≈ 10,000 seconds vs K-Means' 1 second
- Spatial indexing makes it practical for medium n

**Trade-off Summary**:
- Speed: K-Means >> DBSCAN >> GMM
- Flexibility: DBSCAN > GMM >> K-Means
- Interpretability: K-Means ≈ DBSCAN > GMM (harder to interpret covariances)

In [ ]:
# Part 6: Interview Preparation with Visualizations
print(f"\n{'='*80}")
print("INTERVIEW CORNER: DEEP DIVES WITH VISUALIZATIONS")
print(f"{'='*80}")

print("\n" + "="*80)
print("QUESTION 1: Why K-Means Fails on Crescent Data")
print("="*80)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Ground truth moons
ax1 = axes[0]
ax1.scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], c='blue', s=50, alpha=0.7, label='Moon 1 (True)')
ax1.scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], c='red', s=50, alpha=0.7, label='Moon 2 (True)')
ax1.set_xlabel('Feature 1', fontsize=11)
ax1.set_ylabel('Feature 2', fontsize=11)
ax1.set_title('Ground Truth: Non-Convex Moons', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# K-Means result
ax2 = axes[1]
scatter_km = ax2.scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=labels_moons, cmap='viridis', s=50, alpha=0.7)
ax2.scatter(kmeans_moons.cluster_centers_[:, 0], kmeans_moons.cluster_centers_[:, 1],
           c='yellow', marker='*', s=500, edgecolors='black', linewidths=2, label='Centroids')
ax2.set_xlabel('Feature 1', fontsize=11)
ax2.set_ylabel('Feature 2', fontsize=11)
ax2.set_title('K-Means Result: FAILS (Splits Crescents)', fontsize=12, fontweight='bold', color='red')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
plt.colorbar(scatter_km, ax=ax2, label='Cluster')

# Explanation
ax3 = axes[2]
ax3.axis('off')
explanation_text = """WHY K-MEANS FAILS:

1. CENTROID PROBLEM
   • Single point can't
     represent curve shape
   • Points far from center
     still assigned to cluster

2. VARIANCE PROBLEM
   • Curved cluster has high
     within-cluster variance
   • K-Means tries to minimize
     by splitting into spheres

3. ROOT CAUSE
   • Algorithm optimizes only
     for variance, not shape
   • No notion of connectivity

SOLUTION: Use DBSCAN or
Spectral Clustering"""
ax3.text(0.1, 0.5, explanation_text, fontsize=10, family='monospace',
        verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('kmeans_interview_question1.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n→ Visualization 1 saved: K-Means failure on non-convex data")

print(f"\n{'='*80}")
print("QUESTION 2: Computational Complexity Comparison")
print(f"{'='*80}")

# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Algorithm complexity scaling
ax1 = axes[0]
n_points = np.array([1000, 10000, 100000, 1000000])
k_clusters = 10
d_dimensions = 50
i_iterations = 20

kmeans_ops = n_points * k_clusters * d_dimensions * i_iterations / 1e9
gmm_ops = n_points * k_clusters * (d_dimensions**2) * i_iterations / 1e9
dbscan_ops_naive = (n_points**2) / 1e9

ax1.loglog(n_points, kmeans_ops, 'b-o', linewidth=2, markersize=8, label='K-Means O(n·k·d·i)')
ax1.loglog(n_points, gmm_ops, 'r-s', linewidth=2, markersize=8, label='GMM O(n·k·d²·i)')
ax1.loglog(n_points, dbscan_ops_naive, 'g-^', linewidth=2, markersize=8, label='DBSCAN O(n²) naive')
ax1.set_xlabel('Number of Points (n)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Operations (billions)', fontsize=11, fontweight='bold')
ax1.set_title('Algorithm Complexity Scaling', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, which='both', alpha=0.3)

# Practical runtime comparison
ax2 = axes[1]
data_sizes = ['10K\npoints', '100K\npoints', '1M\npoints']
kmeans_times = [0.01, 0.1, 1.0]  # seconds
gmm_times = [0.5, 5, 50]
dbscan_times = [10, 1000, 100000]  # naive DBSCAN - too slow

x = np.arange(len(data_sizes))
width = 0.25

ax2.bar(x - width, kmeans_times, width, label='K-Means', color='steelblue', alpha=0.8)
ax2.bar(x, gmm_times, width, label='GMM', color='coral', alpha=0.8)
ax2.bar(x + width, dbscan_times, width, label='DBSCAN (naive)', color='lightgreen', alpha=0.8)

ax2.set_ylabel('Runtime (seconds, log scale)', fontsize=11, fontweight='bold')
ax2.set_yscale('log')
ax2.set_xticks(x)
ax2.set_xticklabels(data_sizes)
ax2.set_title('Practical Runtime Comparison', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, which='both', alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('kmeans_interview_question2_complexity.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n→ Visualization 2 saved: Complexity comparison")

print(f"\nCONCLUSION:")
print(f"K-Means dominates for large-scale problems due to O(n·k·d·i) complexity.")
print(f"GMM 50-100x slower due to quadratic dimension dependence.")
print(f"DBSCAN practical with spatial indexing; naive O(n²) infeasible for n > 100K.")

## Part 7: Key Takeaways & Cheat Sheet for Placement Interviews

### The 5 Must-Know Concepts

**1. K-Means Minimizes Variance, Not Shape**
- Objective: Minimize WCSS (within-cluster sum of squares)
- Assumes: Spherical, similarly-sized clusters
- Consequence: Fails on non-convex (crescents, rings), overlapping data
- Interview Hook: "It's not about cluster shape, it's about minimizing squared distance"

**2. K-Means++ is Critical for Good Results**
- Random init can trap in poor local optima
- K-Means++ spreads initial centroids: probability ∝ D(x)²
- Empirical: Reduces iterations by 2-5x, improves cluster quality 5-10%
- Interview Hook: "K-Means++ provides O(log K)-approximation vs random's O(K) worst-case"

**3. No Free Lunch: Local Optimum, Not Global**
- Problem is NP-hard in general
- K-Means converges to local minimum guaranteed
- Multiple initializations increase chance of finding better local optima
- Interview Hook: "Always run n_init=10+ and pick best result"

**4. Choosing K Requires Multiple Approaches**
- Elbow: Look for inflection point in inertia vs K
- Silhouette Score: Maximize average silhouette (-1 to 1 range)
- Davies-Bouldin: Minimize ratio of within to between cluster distances
- Gap Statistic: Most principled, compares to random data
- Interview Hook: "Never rely on one metric alone; triangulate with multiple"

**5. Feature Scaling is Non-Negotiable**
- K-Means uses Euclidean distance
- Unscaled features: Large-range features dominate clustering
- Solution: StandardScaler or MinMaxScaler before fitting
- Interview Hook: "Distance-based algorithms need normalized features"

### Quick Reference Table

| Aspect | Answer |
|--------|--------|
| **Time Complexity** | O(n·k·d·i) where i ≈ 10-100 |
| **Space Complexity** | O(n·d + k·d) |
| **Initialization** | K-Means++ (always) over random |
| **Convergence** | Guaranteed to local minimum, not global |
| **Best For** | Large datasets with spherical clusters |
| **Worst For** | Non-convex shapes, very high dimensions |
| **Feature Scaling** | Required (use StandardScaler) |
| **Number of Clusters** | Unknown? Use elbow + silhouette |
| **Outlier Sensitivity** | High (pulled by outliers) |
| **Soft Assignments** | No (use GMM for soft probabilities) |

### Real-World Scenarios

**Use K-Means When:**
- E-commerce: Segment customers by purchase behavior (millions of customers)
- Recommendation: Group products for collaborative filtering
- Image compression: Quantize colors (group pixel values)
- Time series: Detect market regimes in stock data
- Biology: Gene expression clustering (stable, interpretable clusters)

**Don't Use K-Means When:**
- Social networks: Community detection (non-convex topology)
- Anomaly detection: Need explicit outlier labels (DBSCAN better)
- Overlapping clusters: Need soft assignments (GMM better)
- Unknown K: Better with DBSCAN or hierarchical clustering
- Medical diagnosis: Need probabilistic confidence (GMM better)

In [ ]:
# Part 7: Final Summary and Cheat Sheet
print(f"\n{'='*80}")
print("FINAL SUMMARY: K-MEANS CLUSTERING CHEAT SHEET FOR INTERVIEWS")
print(f"{'='*80}")

summary_data = {
    'Time Complexity': 'O(n·k·d·i)',
    'Space Complexity': 'O(n·d + k·d)',
    'Convergence': 'Guaranteed (to local minimum)',
    'Global Optimality': 'NOT guaranteed (NP-hard)',
    'Initialization': 'K-Means++ (critical)',
    'Feature Scaling': 'Required (StandardScaler)',
    'Soft Assignments': 'No (hard clustering only)',
    'Outlier Handling': 'Poor (no explicit detection)',
    'Non-Convex Data': 'Fails (assumes spherical)',
    'Unknown K': 'Use elbow + silhouette methods'
}

for key, value in summary_data.items():
    print(f"{key:<25} : {value}")

print(f"\n{'='*80}")
print("ALGORITHM DECISION TREE")
print(f"{'='*80}")

decision_tree = """
┌─ Is your data VERY LARGE (> 1M points)?
│  ├─ YES → Use K-Means (fastest for large scale)
│  └─ NO → Continue...
│
├─ Do you need to KNOW K beforehand?
│  ├─ YES → Use K-Means directly
│  ├─ NO → Use elbow + silhouette to estimate K, then K-Means
│  └─ NO and want NO parameters → Use DBSCAN
│
├─ Are clusters SPHERICAL and SIMILARLY SIZED?
│  ├─ YES → K-Means is ideal ✓
│  ├─ NO → Consider DBSCAN or Spectral Clustering
│  └─ OVERLAPPING → Use GMM (soft assignments)
│
├─ Do you need PROBABILISTIC assignments?
│  ├─ YES → Use GMM (soft membership)
│  └─ NO → Use K-Means (hard assignment is fine)
│
└─ Do you need to DETECT OUTLIERS?
   ├─ YES → Use DBSCAN (explicit noise labels)
   └─ NO → K-Means or GMM fine
"""
print(decision_tree)

print(f"\n{'='*80}")
print("PERFORMANCE BENCHMARKS (Approximate, depends on hardware)")
print(f"{'='*80}")

benchmark_data = [
    {'n_points': 10000, 'k': 5, 'd': 10, 'kmeans_time_ms': 5, 'gmm_time_ms': 50, 'dbscan_time_ms': 100},
    {'n_points': 100000, 'k': 10, 'd': 20, 'kmeans_time_ms': 50, 'gmm_time_ms': 500, 'dbscan_time_ms': 5000},
    {'n_points': 1000000, 'k': 10, 'd': 50, 'kmeans_time_ms': 500, 'gmm_time_ms': 5000, 'dbscan_time_ms': 'Infeasible'},
]

print(f"\n{'N Points':<12} {'K':<6} {'D':<6} {'K-Means(ms)':<15} {'GMM(ms)':<15} {'DBSCAN(ms)':<15}")
print(f"{'-'*80}")
for bench in benchmark_data:
    print(f"{bench['n_points']:<12} {bench['k']:<6} {bench['d']:<6} {str(bench['kmeans_time_ms']):<15} {str(bench['gmm_time_ms']):<15} {str(bench['dbscan_time_ms']):<15}")

print(f"\n{'='*80}")
print("5 CRITICAL INTERVIEW ANSWERS")
print(f"{'='*80}")

interviews_qa = [
    ("Why K-Means assume spherical clusters?",
     "Because it minimizes Euclidean distance variance. Variance is symmetric around centroid → spherical. Non-spherical requires shape-aware algorithms."),
    
    ("What's the difference K-Means++ vs random init?",
     "K-Means++ spreads initial centroids far apart (prob ∝ D²), avoiding poor local optima. Empirically 5-10x improvement in cluster quality and 2-5x fewer iterations."),
    
    ("How to choose K if unknown?",
     "Use three methods: (1) Elbow method - plot inertia vs K, find knee, (2) Silhouette - maximize average silhouette score, (3) Gap statistic - compare to random data. Triangulate between methods."),
    
    ("Why does K-Means fail on moons/crescents?",
     "Single centroid can't represent curve shape. Points far from center still assigned. To minimize variance, K-Means splits curve into multiple spheres, destroying true structure."),
    
    ("When to use K-Means vs GMM vs DBSCAN?",
     "K-Means: speed critical, large data, spherical clusters. GMM: need soft probabilities, overlapping clusters. DBSCAN: arbitrary shapes, outlier detection, unknown K.")
]

for i, (q, a) in enumerate(interviews_qa, 1):
    print(f"\nQ{i}: {q}")
    print(f"A: {a}")

print(f"\n{'='*80}")
print("✓ COMPLETE K-MEANS GUIDE FINISHED")
print(f"{'='*80}")
print(f"\nYou now understand:")
print(f"  1. Mathematical foundations (objective, convergence, complexity)")
print(f"  2. From-scratch implementation and sklearn optimizations")
print(f"  3. When K-Means works (spherical data) and fails (non-convex)")
print(f"  4. How to select parameters (K, initialization, n_init)")
print(f"  5. Comparison with GMM and DBSCAN")
print(f"\nReady for interview or production deployment! 🚀")